## Building a GPT

Companion notebook to the [Zero To Hero](https://karpathy.ai/zero-to-hero.html) video on GPT.

In [1]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-04-29 10:35:48--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-04-29 10:35:48 (29.2 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  1115394


In [ ]:
# let's look at the first 1000 characters
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [ ]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [ ]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [ ]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [ ]:
print(xb) # our input to the transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        # our input is of the parametres B,T (Batch, current Token)
        # From the embedding table, we will get the probability of each next token given that our current token is the previous token
        # hence, the logits from the embedding table will be B,T, *V for the logits for each token, hence B,T,C


        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)  # pytorch wants that the logits should be flattened, i.e hmare pass har ek position k lie ek logits vector hona chahie
            targets = targets.view(B*T) # pytorch wants outputs in 1d... iska mtlb logits[i] k pass jo Vocab Dimension (C) ka vecotr hoga, uska corresponding target targets[i] hoga and then cross entropy usse niklega
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)  # idx is basically the complete input at once, which is actually 0s
            # we are not really using the loss in this part
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)

            # har batch ka current last token hi dekh rhe hain ham


            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # probailities mil gyi

            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # multinomial lgake probailities k hisab se token nikal gaya. One for each batch



            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
            # us ek ek token ko nikal ke next prediction mei daalte rho, and then again dubara prediction kro

            # [IMP] This is just an untrained model. We are not using losses anywhere


        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
# 8x4 ka 32 bn gya, aur 65 vocab size k hisab se 32x65
print(loss)

print(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist())

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)
[0, 31, 23, 21, 41, 24, 32, 11, 13, 41, 17, 24, 25, 53, 32, 40, 60, 38, 60, 1, 15, 12, 52, 55, 7, 29, 17, 9, 9, 10, 15, 22, 55, 49, 27, 23, 20, 7, 55, 11, 10, 50, 39, 2, 53, 47, 63, 61, 49, 20, 48, 45, 15, 46, 64, 40, 29, 12, 59, 2, 9, 40, 24, 21, 45, 61, 43, 60, 51, 63, 18, 22, 19, 33, 19, 54, 0, 61, 52, 37, 35, 51, 52, 62, 23, 35, 35, 43, 60, 7, 58, 16, 55, 36, 17, 56, 34, 23, 24, 45, 22]

pJ:Bpm&yiltNCjeO3:Cx&vvMYW-txjuAd IRFbTpJ$zkZelxZtTlHNzdXXUiQQY:qFINTOBNLI,&oTigq z.c:Cq,SDXzetn3XVj


In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True) # pytorch will accumulate gradients by default. But this time we need new gradients each and every time, hence doing this
    loss.backward() # backprop, calc grads
    optimizer.step() # update weights

print(loss.item())


4.554598808288574


In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


oTo.JUZ!!zqe!
xBP qbs$Gy'AcOmrLwwt
p$x;Seh-onQbfM?OjKbn'NwUAW -Np3fkz$FVwAUEa-wzWC -wQo-R!v -Mj?,SPiTyZ;o-opr$mOiPJEYD-CfigkzD3p3?zvS;ADz;.y?o,ivCuC'zqHxcVT cHA
rT'Fd,SBMZyOslg!NXeF$sBe,juUzLq?w-wzP-h
ERjjxlgJzPbHxf$ q,q,KCDCU fqBOQT
SV&CW:xSVwZv'DG'NSPypDhKStKzC -$hslxIVzoivnp ,ethA:NCCGoi
tN!ljjP3fwJMwNelgUzzPGJlgihJ!d?q.d
pSPYgCuCJrIFtb
jQXg
pA.P LP,SPJi
DBcuBM:CixjJ$Jzkq,OLf3KLQLMGph$O 3DfiPHnXKuHMlyjxEiyZib3FaHV-oJa!zoc'XSP :CKGUhd?lgCOF$;;DTHZMlvvcmZAm;:iv'MMgO&Ywbc;BLCUd&vZINLIzkuTGZa
D.?


## The mathematical trick in self-attention

In [ ]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"

#

torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
print(x)

tensor([[[ 0.1808, -0.0700],
         [-0.3596, -0.9152],
         [ 0.6258,  0.0255],
         [ 0.9545,  0.0643],
         [ 0.3612,  1.1679],
         [-1.3499, -0.5102],
         [ 0.2360, -0.2398],
         [-0.9211,  1.5433]],

        [[ 1.3488, -0.1396],
         [ 0.2858,  0.9651],
         [-2.0371,  0.4931],
         [ 1.4870,  0.5910],
         [ 0.1260, -1.5627],
         [-1.1601, -0.3348],
         [ 0.4478, -0.8016],
         [ 1.5236,  2.5086]],

        [[-0.6631, -0.2513],
         [ 1.0101,  0.1215],
         [ 0.1584,  1.1340],
         [-1.1539, -0.2984],
         [-0.5075, -0.9239],
         [ 0.5467, -1.4948],
         [-1.2057,  0.5718],
         [-0.5974, -0.6937]],

        [[ 1.6455, -0.8030],
         [ 1.3514, -0.2759],
         [-1.5108,  2.1048],
         [ 2.7630, -1.7465],
         [ 1.4516, -1.5103],
         [ 0.8212, -0.2115],
         [ 0.7789,  1.5333],
         [ 1.6097, -0.4032]]])


In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        #print("xprev ",xprev)
        xbow[b,t] = torch.mean(xprev, 0)
        #print("xbowB,t ", xbow[b,t])



In [ ]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))   # this will create a lower triangular matrix for us with all ones
wei = wei / wei.sum(1, keepdim=True) # this will average out wrt rows, so that if a row had 2 ones, then they will become 0.5 0.5 to help in averaging
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)

print(xbow[0])
print(xbow2[0])
print(torch.allclose(xbow, xbow2))
torch.testing.assert_close(xbow, xbow2)

diff = (xbow - xbow2).abs()
print("max abs diff:", diff.max().item())
print("where:", diff.argmax())
# show full precision
torch.set_printoptions(precision=10)
print(xbow[0, :5, 0])
print(xbow2[0, :5, 0])

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])
False
max abs diff: 3.236345946788788e-08
where: tensor(27)
tensor([ 0.1807715893, -0.0894259512,  0.1489711404,  0.3503567576,
         0.3525155187])
tensor([ 0.1807715893, -0.0894259512,  0.1489711404,  0.3503567576,
         0.3525155187])


In [ ]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))  # lower traingle of 1s
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # -inf on the right traingle
wei = F.softmax(wei, dim=-1) # this will make it the same... like right side zeros and on left matrix 1/num of non zero elements type
xbow3 = wei @ x
torch.allclose(xbow2, xbow3)


True

In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)  # key matrix calculator
query = nn.Linear(C, head_size, bias=False)    # query matrix calculator
value = nn.Linear(C, head_size, bias=False)     # value matrix calculator
k = key(x)   # (B, T, 16)   16 values for each token
q = query(x) # (B, T, 16).   16 values for each token

wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)    Batch x Token(Context) x Token(Context)

# print(wei)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1) #   this is giving it the weights for previous tokens

#print(wei)# basically har batch se har token ka piche se weight

v = value(x)
out = wei @ v
print('out', out)
#out = wei @ x

out.shape

out tensor([[[-1.5713e-01,  8.8009e-01,  1.6152e-01, -7.8239e-01, -1.4289e-01,
           7.4676e-01,  1.0068e-01, -5.2395e-01, -8.8726e-01,  1.9068e-01,
           1.7616e-01, -5.9426e-01, -4.8124e-01, -4.8599e-01,  2.8623e-01,
           5.7099e-01],
         [ 6.7643e-01, -5.4770e-01, -2.4780e-01,  3.1430e-01, -1.2799e-01,
          -2.9521e-01, -4.2962e-01, -1.0891e-01, -4.9282e-02,  7.2679e-01,
           7.1296e-01, -1.1639e-01,  3.2665e-01,  3.4315e-01, -7.0975e-02,
           1.2716e+00],
         [ 4.8227e-01, -1.0688e-01, -4.0555e-01,  1.7696e-01,  1.5811e-01,
          -1.6967e-01,  1.6217e-02,  2.1509e-02, -2.4903e-01, -3.7725e-01,
           2.7867e-01,  1.6295e-01, -2.8951e-01, -6.7610e-02, -1.4162e-01,
           1.2194e+00],
         [ 1.9708e-01,  2.8561e-01, -1.3028e-01, -2.6552e-01,  6.6781e-02,
           1.9535e-01,  2.8073e-02, -2.4511e-01, -4.6466e-01,  6.9287e-02,
           1.5284e-01, -2.0324e-01, -2.4789e-01, -1.6213e-01,  1.9474e-01,
           7.6778e-01],


torch.Size([4, 8, 16])

In [ ]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

# if we dont do this head size, this variance becomes a lot, which means the wei values varies a lot which we dont want
# during initialization, we want that values should be defused..... otherwise all the attention will start coming from only some tokens
# hence, we divide by sqrt(head size)

In [ ]:
k.var()

tensor(1.0449)

In [ ]:
q.var()

tensor(1.0700)

In [ ]:
wei.var()

tensor(1.0918)

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [ ]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [ ]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [ ]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

In [ ]:
# French to English translation example:

# <--------- ENCODE ------------------><--------------- DECODE ----------------->
# les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>



### Full finished code, for reference

You may want to refer directly to the git repo instead though.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 5 # what is the maximum context length for predictions?
max_iters = 200 # kitne iterations tk code chlega
eval_interval = 100# kitni. der baad output print hota rhega
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 64  # dimensios of the embedding layer
n_head = 4    # number of heads of attention
n_layer = 4   # attention for 4 layers
dropout = 0.0
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers

# our tokenizer in this case is character based abd very very simple as explained below

stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
num = 1
def get_batch(split):
    global num
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))  # randomly get Batch_number indices
    x = torch.stack([data[i:i+block_size] for i in ix])       # generate batches using those indices of block_size size
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])    # actual output


    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size))) # kind of just a variable in pytorch

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

      # for parallelly heads, each head will get the same input (B,T,N_Embed)
        B,T,N_Embed = x.shape

        k = self.key(x)   # B,T, N_Embed  * (N_Embed , Head_size) ==== (B,T, Head_Size)
        q = self.query(x) # B,T, N_Embed  * (N_Embed , Head_size) ==== (B,T, Head_Size)

        # compute attention scores ("affinities")

        wei = q @ k.transpose(-2,-1) * head_size**-0.5 # (B, T, N_Embed) @ (B, N_Embed, T) -> (B, T, T) batch x block_size x block_size

        # basically ye affinities aa gyi ek  batch k har token ki apne batch k baaki tokens k saath

        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T).  ye kia because we dont want it to see aagey walie tokens


        wei = F.softmax(wei, dim=-1) # (B, T, T)      # previous weights k saath usko weight dene k lie

        wei = self.dropout(wei) # random dropdouts make it better

        # perform the weighted aggregation of the values
        v = self.value(x)  # B,T, N_Embed  * (N_Embed , Head_size) ==== (B,T, Head_Size)

        out = wei @ v # (B, T, T) @ (B, T, Head_Size) -> (B, T, Head_Size)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1) # we get B,T,Head_Size from each Head, and then we concatenate then to get B,T,N_Embed
        # across the last dimension, bas saari matrices ko chipka de rhe hain. toh head_size*num_heads bn jayega, which is n_embeddings

        projection = self.proj(out) # this is done so that the heads , which are for now just glued, can also talk to each other

        return self.dropout(projection)

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd) # LAyer norm is the equalizer. Baar baar sum k baad bada chota hota jata hai sb isse hi fix hota hai

    def forward(self, x):
        x1 = self.ln1(x) # (B, T, N_Embed) Layer norm does not change the size just fixes and smoothes things

        x = x + self.sa(x1) #(B , T, N_Embed) after the complete multi head attention stuff


        x2 = self.ln2(x) # layer norm  (B , T, N_Embed)

        x = x + self.ffwd(x2) # the OG Feed forward . The conversions are fairly simple here

        return x


        # x = x + self.sa(self.ln1(x))     # residual layer norm is applied before the input goes into self attention and
        # x = x + self.ffwd(self.ln2(x))     # rresidual later norm is applied before it goes to feed forward neural network also
        # return x

        # LayerNorm -> Attention -> LayerNorm -> Feed Forward -> LayerNorm -> Attention -> LayerNorm -> Feedforward

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd) # vocab size transformed to n_embd dimension


        self.position_embedding_table = nn.Embedding(block_size, n_embd) # for each token in our block, its positional encoding is also stored, also transformning to n_embd

        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)]) # n_layer is 4, so there are 4 attention layers

        self.ln_f = nn.LayerNorm(n_embd) # layer norm that is done in the end

        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        # idx is the x parameter, the input
        # target is the y parameter, the output
        # when we do model(x,y), it calls model.forward() directly

        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,N_Embed)
        # so basically from the token embedding table, for each token in idx, we will be getting the corresponding n_embd vector/row making token_embed as (B,T, N_Embed)

        # since there are like 65 different tokens possible(vocab_size, the Embedding is 65 x n_embd)

        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,N_Embed.. torch.arange(T) gives 0,1,2,3.... T)

        x = tok_emb + pos_emb # (B,T,N_Embed) positional embedding is batched/copied automatically to get this

        x = self.blocks(x) # (B,T,N_Embed) # from the attention and FF blocks according to architecture

        x = self.ln_f(x) # (B,T,N_Embed)

        logits = self.lm_head(x) # (B,T, N_Embed ) * (N_Embd , Vocab_size) = (B , T, Vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, V_size = logits.shape
            logits = logits.view(B*T, V_size) # because pytorch needs without batching to calculate outputs
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            print(idx_cond)
            # get the predictions
            logits, loss = self(idx_cond) # since targets is none, so no loss
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes Only C. We take the logits of only the last one
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # Only C since there is no batch
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (T+1), no batch here
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
# for name, p in m.named_parameters():
#     print(f"{name:45} {tuple(p.shape)} {p.numel():,}")
#print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')


# calculating the number of params manually down below

head_size = n_embd // n_head

blockParams  = n_embd* n_embd + n_embd  +  n_embd* n_embd * 4 + 4*n_embd + n_embd*4*n_embd + n_embd+  2*n_embd + 2*n_embd + 4*(3*n_embd*head_size)

calculatedParams =   vocab_size * n_embd + block_size * n_embd + n_layer*(blockParams) + 2*n_embd + n_embd * vocab_size + vocab_size

# paramateres in nn.linear(a,b) = a*b + b (last b for bias)
# parameteres in nn.LayerNorm(b) are 2*b
# parameteres in nn.Embedding(a,b) are a*b


print(' num2 ', calculatedParams)


# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        # bs print krne k lie training and validation dono pr loss nikal rhe hain

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


 num2  208001
step 0: train loss 4.4609, val loss 4.4453
step 100: train loss 3.0608, val loss 3.0559
step 199: train loss 2.7858, val loss 2.7938
tensor([[0]])
tensor([[ 0, 28]])
tensor([[ 0, 28, 46]])
tensor([[ 0, 28, 46, 47]])
tensor([[ 0, 28, 46, 47, 12]])
tensor([[28, 46, 47, 12,  1]])
tensor([[46, 47, 12,  1, 61]])
tensor([[47, 12,  1, 61, 43]])
tensor([[12,  1, 61, 43,  1]])
tensor([[ 1, 61, 43,  1, 58]])
tensor([[61, 43,  1, 58, 46]])
tensor([[43,  1, 58, 46, 43]])
tensor([[ 1, 58, 46, 43, 52]])
tensor([[58, 46, 43, 52, 42]])
tensor([[46, 43, 52, 42, 54]])
tensor([[43, 52, 42, 54,  1]])
tensor([[52, 42, 54,  1, 44]])
tensor([[42, 54,  1, 44, 10]])
tensor([[54,  1, 44, 10, 53]])
tensor([[ 1, 44, 10, 53, 12]])
tensor([[44, 10, 53, 12, 21]])
tensor([[10, 53, 12, 21,  0]])
tensor([[53, 12, 21,  0, 30]])
tensor([[12, 21,  0, 30, 20]])
tensor([[21,  0, 30, 20,  0]])
tensor([[ 0, 30, 20,  0, 24]])
tensor([[30, 20,  0, 24,  7]])
tensor([[20,  0, 24,  7, 25]])
tensor([[ 0, 24,  7, 25,  

KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn.functional as F

# corpus
sentences = [
    "<s> I am Sam </s>",
    "<s> Sam I am </s>",
    "<s> I do not like Sam </s>"
]

# build vocab
tokens = sorted({w for s in sentences for w in s.split()})
stoi = {w:i for i,w in enumerate(tokens)}
itos = {i:w for w,i in stoi.items()}
V = len(tokens)

print("stoi", stoi)
print("itos", itos)

# count bigrams
counts = torch.zeros(V, V, dtype=torch.float32)
for s in sentences:
    ws = s.split()
    for w1, w2 in zip(ws, ws[1:]):
        print(w1,' ',w2)
        counts[stoi[w1], stoi[w2]] += 1

# add-one smoothing so unseen pairs are possible
counts += 1


print('counts= ',  counts)

# turn counts into probabilities
probs = counts / counts.sum(dim=1, keepdim=True) # shape [V, V]


print('probs= ' , probs)

# sample
def sample(start="<s>", max_len=10):
    idx = stoi[start]
    out = []
    for _ in range(max_len):
        idx = torch.multinomial(probs[idx], 1).item() # sample from row... the one with higher value has higher probability of coming
        w = itos[idx]
        if w == "</s>": break
        out.append(w)
    return " ".join(out)

print("P(am | I) =", probs[stoi["I"], stoi["am"]].item())
for _ in range(5):
    print(">", sample())

In [ ]:
import torch.nn as nn

# prepare training pairs (prev -> next)
xs, ys = [], []
for s in sentences:
    ws = s.split()
    for w1, w2 in zip(ws, ws[1:]):
        xs.append(stoi)
        ys.append(stoi)
xs = torch.tensor(xs)
ys = torch.tensor(ys)

# model: each row is logits for next token
logits_table = nn.Embedding(V, V) # initialized randomly
optimizer = torch.optim.AdamW(logits_table.parameters(), lr=0.1)

for step in range(200):
    logits = logits_table(xs) # [N, V]
    loss = F.cross_entropy(logits, ys)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# use learned probabilities
with torch.no_grad():
    learned_probs = F.softmax(logits_table.weight, dim=1)

def sample_learned():
    idx = stoi["<s>"]
    out = []
    for _ in range(10):
        idx = torch.multinomial(learned_probs[idx], 1).item()
        w = itos[idx]
        if w == "</s>": break
        out.append(w)
    return " ".join(out)

print("loss:", loss.item())
for _ in range(5):
    print(">", sample_learned())